# Operators and Expressions

**Topic:** `1.6` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*Reference implementations for all 10 exercises plus the robotics challenge.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

> **Compare, do not copy**
>
> Work each exercise yourself first. When you open this notebook, compare your approach with the reference: check the reasoning, the edge cases and the complexity. If your solution differs, that is fine — verify it with the tests provided and keep whichever is clearer.

## Solution 1 — Divide without raising on zero

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** arithmetic, conditionals, defaults

**Explanation and reasoning**

The zero check must come before the division, and the type check must come before both so a string produces a clear TypeError rather than an obscure one. Rejecting bool explicitly keeps True from being used as 1.

In [ ]:
def safe_divide(numerator, denominator, default=0.0):
    for value in (numerator, denominator):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f'not a number: {value!r}')
    if denominator == 0:
        return default
    return numerator / denominator

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

A negative denominator divides normally; a fractional divisor works too.

**Alternative approaches**

Using fractions.Fraction avoids the float result entirely.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert safe_divide(10, 4) == 2.5
assert safe_divide(1, 0) == 0.0
assert safe_divide(1, 0, default=None) is None

---

## Solution 2 — Test a value against an inclusive range

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** comparisons, chaining

**Explanation and reasoning**

The chained comparison is a single expression that the interpreter rewrites into a method-chained test, so it short-circuits and cannot evaluate the second comparison when the first already fails. The type guard is required because comparing a string with a number raises.

In [ ]:
def in_range(value, low, high) -> bool:
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        return False
    return low <= value <= high

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

low equal to high accepts only that exact value.

**Alternative approaches**

Swapping the bounds first makes the function total for reversed input.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert in_range(50, 0, 100) is True
assert in_range(0, 0, 100) is True
assert in_range('50', 0, 100) is False

---

## Solution 3 — Choose a battery verdict

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** conditional expression, comparisons

**Explanation and reasoning**

The two boundary tests are ordered from most restrictive to least, so the first one that matches decides the label. Returning a string rather than printing keeps the function usable in a larger expression.

In [ ]:
def battery_verdict(pct) -> str:
    return (
        'LOW'
        if pct < 20
        else 'MEDIUM' if pct < 50 else 'OK'
    )

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

Exactly 20 falls into MEDIUM because the test is strictly less than 20.

**Alternative approaches**

A dict of thresholds with a loop is clearer for five or more bands.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert battery_verdict(35) == 'MEDIUM'
assert battery_verdict(10) == 'LOW'
assert battery_verdict(50) == 'OK'
assert battery_verdict(20) == 'MEDIUM'

---

## Solution 4 — Count vowels in a label

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** strings, in, membership

**Explanation and reasoning**

A frozenset gives O(1) membership tests, so the cost is linear in the length of the string. Lower-casing first means only lowercase letters need to be listed, and the generator expression avoids building an intermediate list.

In [ ]:
VOWELS = frozenset('aeiou')


def count_vowels(text: str) -> int:
    return sum(1 for ch in text.lower() if ch in VOWELS)

**Complexity**

Time O(n); space O(1).

**Edge cases and failure modes**

Digits and punctuation are simply not in the set, so they are skipped.

**Alternative approaches**

len(text) - count of consonants is equivalent but more work.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert count_vowels('Rover-01') == 2
assert count_vowels('') == 0
assert count_vowels('AEIOU') == 5

---

## Solution 5 — Average readings, ignoring junk

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** comprehensions, builtins, validation

**Explanation and reasoning**

Filtering first means sum never sees a value it cannot add, so the function is total over any input list. Returning None for an empty or fully filtered result distinguishes 'no data' from 'an average of zero', which is a distinction a caller genuinely needs.

In [ ]:
def safe_divide(numerator, denominator, default=None):
    if denominator == 0:
        return default
    return numerator / denominator


def average(values) -> float | None:
    numbers = [
        v for v in values
        if isinstance(v, (int, float)) and not isinstance(v, bool)
    ]
    if not numbers:
        return None
    return safe_divide(sum(numbers), len(numbers))

**Complexity**

Time O(n); space O(n) for the filtered list.

**Edge cases and failure modes**

A list of only strings returns None rather than raising.

**Alternative approaches**

A generator with sum() avoids the intermediate list at the cost of a separate length.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert average([1.0, 2.0, 'x', 3.0]) == 2.0
assert average([]) is None
assert average(['a']) is None
assert average([True, 2.0]) == 2.0

---

## Solution 6 — Decode a packed status word

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** bitwise, dicts, validation

**Explanation and reasoning**

Folding every known mask together gives the set of bits this decoder understands, so ANDing its complement with the word isolates exactly the bits that are set but unrecognised. Reporting them is what turns a silent interoperability failure into a visible warning.

In [ ]:
def decode_status(word: int, known: dict) -> dict:
    if isinstance(word, bool) or not isinstance(word, int):
        raise TypeError('status word must be an int')
    if word < 0:
        raise ValueError('status word must not be negative')
    active = [name for name, mask in known.items() if word & mask]
    defined = 0
    for mask in known.values():
        defined |= mask
    return {
        'word': word,
        'flags': active,
        'unknown_bits': word & ~defined,
    }

**Complexity**

Time O(k) for k known flags; space O(k).

**Edge cases and failure modes**

Zero gives an empty flag list and zero unknown bits.

**Alternative approaches**

A dataclass return type is clearer in a larger codebase.

**Tests — run the cell to verify the reference solution**

In [ ]:
KNOWN = {'CHARGING': 1, 'LOW_BATTERY': 2, 'OBSTACLE': 4, 'ESTOP': 8}
out = decode_status(0b0011, KNOWN)
assert out['flags'] == ['CHARGING', 'LOW_BATTERY']
assert decode_status(0b10000, KNOWN)['unknown_bits'] == 16

---

## Solution 7 — Encode flag names into a word

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** bitwise, dicts, sets

**Explanation and reasoning**

Folding the masks together with OR is commutative and idempotent, so duplicate names are harmless and the caller's order does not affect the result. Validating inside the loop means the error names the offending flag rather than the whole batch.

In [ ]:
def encode_flags(names, known: dict) -> int:
    word = 0
    for name in names:
        if name not in known:
            raise ValueError(f'unknown flag: {name!r}')
        word |= known[name]
    return word

**Complexity**

Time O(n) for n names; space O(1).

**Edge cases and failure modes**

An empty name list returns 0, which is the natural empty word.

**Alternative approaches**

Using functools.reduce with operator.or_ is shorter but cannot validate.

**Tests — run the cell to verify the reference solution**

In [ ]:
KNOWN = {'CHARGING': 1, 'LOW_BATTERY': 2, 'OBSTACLE': 4, 'ESTOP': 8}
assert encode_flags(['OBSTACLE', 'CHARGING'], KNOWN) == 5
assert encode_flags([], KNOWN) == 0
assert encode_flags(['CHARGING', 'CHARGING'], KNOWN) == 1

---

## Solution 8 — Set, clear and test a single flag

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** bitwise, tuples, composition

**Explanation and reasoning**

Each function is the single operator whose name describes it, which makes the set of four read as a specification. Clearing uses the complement of the mask so only the named bits are zeroed and every other bit in the word is preserved exactly.

In [ ]:
def set_flag(word: int, mask: int) -> int:
    return word | mask


def clear_flag(word: int, mask: int) -> int:
    return word & ~mask


def toggle_flag(word: int, mask: int) -> int:
    return word ^ mask


def test_flag(word: int, mask: int) -> bool:
    return bool(word & mask)

**Complexity**

Time O(1) for all four; space O(1).

**Edge cases and failure modes**

Clearing a bit that is not set returns the word unchanged.

**Alternative approaches**

A class wrapping the word gives the same four operations with less call-site noise.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert clear_flag(0b1111, 0b0010) == 13
assert set_flag(0b1111, 0b10000) == 31
assert toggle_flag(toggle_flag(5, 2), 2) == 5
assert test_flag(0b0010, 0b0010) is True

---

## Solution 9 — Evaluate a simple arithmetic expression safely

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** ast, recursion, validation

**Explanation and reasoning**

Walking the tree means only the node types we explicitly handle can be evaluated, so a call, an attribute access or a name is rejected rather than executed. The operator dispatch table is the complete allow-list, which is what makes this safe where eval would not be.

In [ ]:
import ast

OPS = {
    ast.Add: lambda a, b: a + b,
    ast.Sub: lambda a, b: a - b,
    ast.Mult: lambda a, b: a * b,
    ast.Div: lambda a, b: a / b,
    ast.FloorDiv: lambda a, b: a // b,
    ast.Mod: lambda a, b: a % b,
    ast.Pow: lambda a, b: a ** b,
}


def _value(node):
    if isinstance(node, ast.Constant):
        if isinstance(node.value, bool) or not isinstance(node.value, (int, float)):
            raise ValueError('only numeric literals are allowed')
        return node.value
    if isinstance(node, ast.UnaryOp):
        operand = _value(node.operand)
        return -operand if isinstance(node.op, ast.USub) else operand
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        left, right = _value(node.left), _value(node.right)
        try:
            return OPS[type(node.op)](left, right)
        except ZeroDivisionError as exc:
            raise ValueError('division by zero') from exc
    raise ValueError(f'unsupported expression node: {type(node).__name__}')

**Complexity**

Time O(n) in the number of nodes; space O(d) for tree depth.

**Edge cases and failure modes**

Division by zero is converted into a ValueError so callers see one failure type.

**Alternative approaches**

ast.literal_eval handles only literals, not arithmetic.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert _value(ast.parse('2 + 3 * 4', mode='eval').body) == 14
assert _value(ast.parse('-5 + 1', mode='eval').body) == -4

---

## Solution 10 — Report how an expression was grouped

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** ast, precedence, reporting

**Explanation and reasoning**

Appending the parent before recursing produces pre-order, so the report reads from the outermost operation inwards. ast.unparse renders each sub-expression exactly as the parser understood it, which is what makes the grouping visible without evaluating anything.

In [ ]:
import ast

SYMBOLS = {
    ast.Add: '+',
    ast.Sub: '-',
    ast.Mult: '*',
    ast.Div: '/',
    ast.FloorDiv: '//',
    ast.Mod: '%',
    ast.Pow: '**',
}


def precedence_report(text: str) -> list[tuple]:
    report: list[tuple] = []

    def walk(node):
        if isinstance(node, ast.BinOp) and type(node.op) in SYMBOLS:
            report.append((
                SYMBOLS[type(node.op)],
                ast.unparse(node.left),
                ast.unparse(node.right),
            ))
            walk(node.left)
            walk(node.right)

    walk(ast.parse(text, mode='eval').body)
    return report

**Complexity**

Time O(n) in the tree size; space O(d) for depth.

**Edge cases and failure modes**

A bare literal has no BinOp nodes, so the report is empty.

**Alternative approaches**

Printing the tree with ast.dump is faster but far less readable for teaching.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert precedence_report('1 + 2 * 3') == [('+', '1', '2 * 3'), ('*', '2', '3')]
assert precedence_report('1') == []

---

# Robotics challenge solution — ROBO-X Challenge: Status Word Controller

**Explanation**

Folding each condition into the word with OR means the caller receives one value they can forward to firmware unchanged. Treating a sensor read error as an obstacle is the safety-critical decision here: an unknown distance is treated as the dangerous case, never as a clear path.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'shared').is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root))

from shared.robo_x_sim import SimulatedRobot, SensorError  # noqa: E402

CHARGING, LOW_BATTERY, OBSTACLE, ESTOP = 1, 2, 4, 8
KNOWN = {'CHARGING': CHARGING, 'LOW_BATTERY': LOW_BATTERY, 'OBSTACLE': OBSTACLE, 'ESTOP': ESTOP}


def read_status(robot, obstacle_threshold: float = 1.0) -> dict:
    """Build a status word from live telemetry, without raising."""
    state = robot.status()
    word = 0
    if state['battery_pct'] < 20:
        word |= LOW_BATTERY
    try:
        if robot.read_sensor('distance') < obstacle_threshold:
            word |= OBSTACLE
    except SensorError:
        word |= OBSTACLE          # an unreadable sensor is not a clear path
    active = [name for name, mask in KNOWN.items() if word & mask]
    return {'word': word, 'flags': active, 'battery_pct': state['battery_pct']}

**Complexity**

Time O(1) for a fixed number of flags; space O(1).

**Notes and trade-offs**

A faulty distance sensor sets OBSTACLE rather than silently clearing it.

In [ ]:
robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
out = read_status(robot)
assert out['word'] == 0
assert out['flags'] == []

## Reflection

For each exercise where your solution differed from the reference, write one sentence explaining *why* yours was better or worse. This is the highest-value part of the notebook:

- A case the reference handles but mine did not.
- A case mine handles better — and what the reference is missing.
- A performance difference that matters at robot scale.